# C1 classical: full run on a Colab T4

Kernel: **Colab, T4 GPU** (Runtime → Change runtime type → T4). Run all cells top to bottom.

Re-runs every tuning stage (s1–s4, 243 configs × seeds 0–4) and the `best` / `fast` configs on the report seeds (10–19), all tagged `hw=colab-t4`, so every time in the report comes from the same GPU type as the rest of the team. Expect roughly 30–60 min (Colab has only 2 CPU cores).

**Resumable:** rows are logged to a copy of `results/massimo.csv` on your Google Drive, so they survive a disconnect or a runtime reset. Just run all cells again: rows already logged for `hw=colab-t4` are skipped. The last code cell merges the Drive copy back into the repo.

If you re-run after a change, **restart the kernel first** (`common.data` only reads `COMP433_DATA` once, at import).

In [8]:
# 1. Get the code onto the Colab machine
import os
import shutil
from getpass import getpass

REPO = "/content/repo"
BRANCH = "massimo/c1"

if not os.path.exists(REPO):
    token = getpass("GitHub token: ")
    !git clone -q -b {BRANCH} https://{token}@github.com/MassPaol/comp433-final-project.git {REPO}
    del token
else:
    !git -C {REPO} pull -q

%cd {REPO}
!pip install -q -e . -r requirements.txt

# Commit identity for saving results at the end (a fresh Colab VM has none)
!git -C {REPO} config user.name "Massimo Paolini"
!git -C {REPO} config user.email "mpaolini119@gmail.com"

/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for comp433 (pyproject.toml) ... done


In [9]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# 3. Copy CIFAR-10 from the team folder to the Colab machine (faster than reading from Drive)
DRIVE_DATA = "/content/drive/MyDrive/comp433_cache/data"  # needs the comp433_cache shortcut in My Drive
LOCAL_DATA = "/content/data"

if not os.path.exists(f"{LOCAL_DATA}/cifar-10-python.tar.gz"):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    shutil.copy(f"{DRIVE_DATA}/cifar-10-python.tar.gz", LOCAL_DATA)  # torchvision unpacks it on first load

os.environ["COMP433_DATA"] = LOCAL_DATA

In [ ]:
# 4. Import the project code (must come after COMP433_DATA is set) and check the data loads
import torch
from common import harness, data

assert data.DATA_ROOT == os.environ["COMP433_DATA"], (
    "common.data was imported before COMP433_DATA was set. Restart the kernel and run all cells again."
)
print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "none (running on CPU)")

X, y, Xv, yv, classes = data.load_instance(0)
print(X.shape, Xv.shape, [data.CLASS_NAMES[c] for c in classes])  # expect (100, 32, 32, 3) (700, 32, 32, 3) ['class_1', 'class_2']

GPU: Tesla T4
(100, 32, 32, 3) (700, 32, 32, 3) ['bird', 'ship']


In [12]:
# 5. Make sure this really is a T4 (every row gets tagged hw=colab-t4), and log to Drive
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(), (
    f"Expected a T4 GPU, got {torch.cuda.get_device_name() if torch.cuda.is_available() else 'no GPU'}")
HW = "colab-t4"

RESULTS_DIR = "/content/drive/MyDrive/comp433_c1_classical_results"
RESULTS = f"{RESULTS_DIR}/massimo.csv"
os.makedirs(RESULTS_DIR, exist_ok=True)
if not os.path.exists(RESULTS):
    shutil.copy("results/massimo.csv", RESULTS)  # start from the repo's rows
print("logging to", RESULTS)

logging to /content/drive/MyDrive/comp433_c1_classical_results/massimo.csv


In [13]:
# 6. All tuning stages on seeds 0-4 (prints one line per config)
for stage in ["s1", "s2", "s3", "s4"]:
    !python -W ignore -m c1.classical.run --stage {stage} --seeds tune --hw {HW} --results-dir {RESULTS_DIR}

[1/42] s1_raw_lr: acc 0.7423 +/- 0.0563 | fit 0.09s | seeds [0, 1, 2, 3, 4]
[2/42] s1_raw_linsvm: acc 0.7343 +/- 0.0584 | fit 0.10s | seeds [0, 1, 2, 3, 4]
[3/42] s1_raw_rbfsvm: acc 0.8000 +/- 0.0394 | fit 0.02s | seeds [0, 1, 2, 3, 4]
[4/42] s1_raw_rf: acc 0.7791 +/- 0.0447 | fit 1.62s | seeds [0, 1, 2, 3, 4]
[5/42] s1_raw_hgb: acc 0.7506 +/- 0.0420 | fit 1.46s | seeds [0, 1, 2, 3, 4]
[6/42] s1_raw_xgb: acc 0.7440 +/- 0.0505 | fit 2.37s | seeds [0, 1, 2, 3, 4]
[7/42] s1_raw_knn: acc 0.6440 +/- 0.0800 | fit 0.01s | seeds [0, 1, 2, 3, 4]
[8/42] s1_rawpca_lr: acc 0.7237 +/- 0.0479 | fit 0.37s | seeds [0, 1, 2, 3, 4]
[9/42] s1_rawpca_linsvm: acc 0.7180 +/- 0.0486 | fit 0.47s | seeds [0, 1, 2, 3, 4]
[10/42] s1_rawpca_rbfsvm: acc 0.7906 +/- 0.0452 | fit 0.09s | seeds [0, 1, 2, 3, 4]
[11/42] s1_rawpca_rf: acc 0.7703 +/- 0.0406 | fit 1.21s | seeds [0, 1, 2, 3, 4]
[12/42] s1_rawpca_hgb: acc 0.7300 +/- 0.0521 | fit 0.15s | seeds [0, 1, 2, 3, 4]
[13/42] s1_rawpca_xgb: acc 0.7409 +/- 0.0411 | fit

In [14]:
# 7. Re-apply the selection rule with T4 times, and check it agrees with FINAL in configs.py
!python -m c1.classical.select --hw {HW} --results {RESULTS}

import pandas as pd
from c1.classical import configs, select as sel

summary = sel.summarize_configs(pd.read_csv(RESULTS), data.TUNE_SEEDS, HW)
summary = summary[summary.index.str.match(r"s[1-4]_")]
assert len(summary) == 243, f"expected 243 complete tuning configs, found {len(summary)}"
best, fast = sel.select(summary)
picked = {"best": best, "fast": fast}
if picked != configs.FINAL_SOURCE:
    raise RuntimeError(
        f"T4 timings pick {picked}, but configs.FINAL_SOURCE is {configs.FINAL_SOURCE}. "
        "Update FINAL_SOURCE in c1/classical/configs.py (and the selection entry in NOTES.md), "
        "push, restart the kernel and run all cells again.")
print("T4 selection agrees with configs.FINAL_SOURCE:", picked)

                                        acc_mean  acc_std  time_mean  n
config_id                                                              
s4_scat+hog+hsv_lrcv_fliptrain_fliptta    0.8997   0.0317     5.9686  5
s4_scat+hog+hsv_lrcv_fliptrain            0.8983   0.0325     5.6456  5
s4_scat+hog+hsv_lrcv_fliptta              0.8903   0.0328     2.6224  5
s4_scat_fliptrain_fliptta                 0.8891   0.0356     5.6226  5
s4_hog+hsv_rbfsvm_fliptrain_fliptta       0.8857   0.0420     1.0574  5
s4_hog+hsv_rbfsvm_fliptrain               0.8854   0.0440     0.5904  5
s4_scat_fliptrain                         0.8851   0.0357     5.2322  5
s4_hog+hsv_rbfsvm_fliptta                 0.8834   0.0429     1.3072  5
s4_scat+hog+hsv_lrcv                      0.8803   0.0331     1.9529  5
s4_scat_fliptta                           0.8794   0.0341     1.2542  5
s4_scat+hog_lrcv                          0.8791   0.0349     1.5067  5
s4_hog+hsv_rbfsvm                         0.8783   0.0478     0.

In [15]:
# 8. best and fast on the report seeds 10-19
!python -W ignore -m c1.classical.run --config best fast --seeds report --hw {HW} --results-dir {RESULTS_DIR}

[1/2] best: acc 0.8509 +/- 0.0822 | fit 5.01s | seeds [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[2/2] fast: acc 0.8380 +/- 0.0799 | fit 0.09s | seeds [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [16]:
# 9. Report-seed summary (numbers for the report)
from scipy.stats import wilcoxon
from c1.classical.run import hw_of, own_rows

df = own_rows(pd.read_csv(RESULTS))
r = df[df.config_id.isin(["best", "fast"]) & df.seed.isin(data.REPORT_SEEDS) & (df.notes.map(hw_of) == HW)]
w = r.pivot_table(index=["seed", "class_pair"], columns="config_id",
                  values=["val_acc", "train_time_s", "infer_time_s"])
for c in ["best", "fast"]:
    acc = w["val_acc"][c] * 100
    infer = w["infer_time_s"][c].mean()
    print(f"{c}: {acc.mean():.2f} +/- {acc.std(ddof=0):.2f} %   train {w['train_time_s'][c].mean():.3f} s   "
          f"inference {infer:.3f} s / 700 img ({infer / 700 * 1000:.2f} ms/img)")
d = (w["val_acc"]["best"] - w["val_acc"]["fast"]) * 100
print(f"paired best - fast: {d.mean():+.2f} pp, best wins {(d > 0).sum()}/{len(d)}, "
      f"Wilcoxon p = {wilcoxon(d).pvalue:.4f}")

best: 85.09 +/- 8.22 %   train 5.005 s   inference 0.689 s / 700 img (0.98 ms/img)
fast: 83.80 +/- 7.99 %   train 0.095 s   inference 0.451 s / 700 img (0.64 ms/img)
paired best - fast: +1.29 pp, best wins 8/10, Wilcoxon p = 0.0332


In [17]:
# 10. Regenerate the report figures from the T4 rows
!python -m c1.classical.figures --hw {HW} --results {RESULTS}

wrote report/figures/c1_classical_s1_heatmap.pdf
wrote report/figures/c1_classical_s1_heatmap.png
wrote report/figures/c1_classical_sweeps.pdf
wrote report/figures/c1_classical_sweeps.png
wrote report/figures/c1_classical_acc_vs_time.pdf
wrote report/figures/c1_classical_acc_vs_time.png
wrote report/figures/c1_classical_report_pairs.pdf
wrote report/figures/c1_classical_report_pairs.png


In [18]:
# 11. Merge the Drive copy back into the repo's results file
from c1.classical.run import merge_results
print("added", merge_results(RESULTS, "results/massimo.csv"), "rows to results/massimo.csv")

added 1235 rows to results/massimo.csv


In [19]:
# 12. Save the results

!git add results/massimo.csv report/figures/c1_classical_*
!git commit -m "Run C1 classical on Colab T4"
!git push

[massimo/c1 4f08455] Run C1 classical on Colab T4
 9 files changed, 1235 insertions(+)
fatal: could not read Password for 'https://github.com': No such device or address
